# 원본 실험 기록
원본 파일: Music_gen/Load_model_and_create_midi_file.ipynb

실행 출력과 메타데이터를 제거하고 Drive 경로를 /content/project로 치환했습니다. 셀 순서와 원본 코드를 보존하므로 위에서 아래로 실행이 보장되지 않습니다. 정리된 실행 흐름은 상위 폴더의 01–05 노트북을 참고하세요.


DCGAN 모델 및 학습 루프는 TensorFlow Authors의 DCGAN 튜토리얼을 바탕으로
음악 이미지에 맞게 수정한 프로젝트 코드입니다. Copyright 2019 The TensorFlow Authors.
해당 기반 코드에는 Apache License 2.0이 적용됩니다. 저장소의 THIRD_PARTY_NOTICES.md와
LICENSES/Apache-2.0.txt를 참고하세요. 공개용 정리 과정에서 경로·셀 순서·설명을 수정했습니다.

In [ ]:
!pip install mido
!pip install midiutil

In [ ]:
import numpy as np
from google.colab import drive
drive.mount('/content/drive/')

In [ ]:
import midiutil
def write_midi(seq, bpm, path):
    mf = midiutil.MIDIFile(1, file_format=1)
    track = 0
    channel = 0
    mf.addTempo(track, 0, bpm)
    for i, n in enumerate(seq):
        mf.addNote(
            track,
            channel,
            int(n['pitch']),
            n['start_time'],
            n['length'],
            int(n['velocity'])
        )
    with open(path, 'wb') as outf:
        mf.writeFile(outf)

In [ ]:
import matplotlib.pyplot as plt
data = np.load('/content/project/Data/data_set/data_set.npy')
plt.imshow(data[182], cmap='gray', origin='lower')

In [ ]:
data = data.reshape(data.shape[0], 64, 64, 1).astype('float32')
data.shape

In [ ]:
reshaped_data = data[182].reshape(64,64)

In [ ]:
reshaped_data

In [ ]:
#피치 빈칸 삽입

reshaped_data2 = reshaped_data

for i in range(36):
  reshaped_data2 = np.insert(reshaped_data2,0,0, axis=0)
for i in range(12):
  reshaped_data2 = np.insert(reshaped_data2,60,0, axis=0)
for i in range(16):
  reshaped_data2 = np.insert(reshaped_data2,112,0, axis=0)
  

reshaped_data2.shape

In [ ]:
sequence = []
stack = 0
for pit, note in enumerate(reshaped_data2):
  for i,j in enumerate(note):
    if j != 0 and i != 63:
      if note[i+1] != 0:
        stack = stack + 1
        if i+1 == 63: #다음 노트가 맨 뒤 노트일때
          sequence.append({
          'pitch' : pit,
          'start_time' : (i+1-stack)/4,
          'length' : (stack+1)*0.25,
          'velocity' : 80
          })
          stack=0
      else:
        sequence.append({
          'pitch' : pit,
          'start_time' : (i-stack)/4,
          'length' : (stack+1)*0.25,
          'velocity' : 80
          })
        stack=0
    elif j !=0 and i == 63 and note[i-1] == 0:
      sequence.append({
          'pitch' : pit,
          'start_time' : i/4,
          'length' : 0.25,
          'velocity' : 80
        })
      stack=0

sequence

In [ ]:
sequence = sorted(sequence, key=lambda k: k['start_time'])
sequence

In [ ]:
write_midi(sequence, 120, '/content/project/dfaaaas.mid')